In [1]:
!pip install gensim nltk -q

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

import pandas as pd
import numpy as np
import pickle, os
from pathlib import Path
from huggingface_hub import HfApi, HfFileSystem

HF_REPO = "vngclinh/goodreads-preprocessed"
hf_fs = HfFileSystem(token=HF_TOKEN)
api = HfApi()

os.makedirs("/kaggle/working/lda", exist_ok=True)

In [2]:
import nltk, re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

STOP = set(stopwords.words("english"))
lemma = WordNetLemmatizer()

def tokenize(text):
    text = re.sub(r"[^a-zA-Z\s]", " ", str(text).lower())
    tokens = text.split()
    return [lemma.lemmatize(t) for t in tokens
            if t not in STOP and len(t) > 2]

# Load từ HF
with hf_fs.open(f"datasets/{HF_REPO}/lda_corpus.parquet", "rb") as f:
    df = pd.read_parquet(f)

print(f"Corpus size: {len(df):,} rows")
print(df.dtypes)
print(df["review_text"].iloc[0][:200])  # verify raw text

df["tokens"] = df["review_text"].map(tokenize)
df = df[df["tokens"].str.len() >= 5]
print(f"After tokenize: {len(df):,}")

Corpus size: 1,240,635 rows
review_id         object
user_id           object
book_id           object
primary_genre     object
rating             int32
review_text       object
vote_weight      float64
edge_weight      float64
dtype: object
A "praise" book, the premise of this picture book helps open our eyes to seeing the beauty of God's creation in various weather and circumstances. Like with All of Me That You Can't See, the syllabic 
After tokenize: 1,237,163


In [3]:
from gensim import corpora

dictionary = corpora.Dictionary(df["tokens"])
print(f"Raw vocab: {len(dictionary):,}")

dictionary.filter_extremes(no_below=20, no_above=0.4, keep_n=50_000)
print(f"Filtered vocab: {len(dictionary):,}")

corpus = [dictionary.doc2bow(tok) for tok in df["tokens"]]

dictionary.save("/kaggle/working/lda/dict.gensim")
corpora.MmCorpus.serialize("/kaggle/working/lda/corpus.mm", corpus)
print("Saved dictionary + corpus")

Raw vocab: 1,486,595
Filtered vocab: 50,000
Saved dictionary + corpus


In [4]:
from gensim.models import LdaMulticore, CoherenceModel

# Thử 3 giá trị, chọn coherence cao nhất
results = []
for n in [30, 40]:
    print(f"\nTraining LDA k={n}...")
    lda = LdaMulticore(
        corpus=corpus,
        id2word=dictionary,
        num_topics=n,
        workers=3,
        passes=10,
        chunksize=10_000,
        random_state=42
    )
    cm = CoherenceModel(
        model=lda, texts=df["tokens"].tolist(),
        dictionary=dictionary, coherence="c_v"
    )
    score = cm.get_coherence()
    results.append((n, score, lda))
    print(f"  k={n} coherence={score:.4f}")

best_n, best_score, best_lda = max(results, key=lambda x: x[1])
print(f"\nBest: k={best_n}, coherence={best_score:.4f}")
best_lda.save(f"/kaggle/working/lda/lda_{best_n}.model")


Training LDA k=30...
  k=30 coherence=0.5591

Training LDA k=40...
  k=40 coherence=0.5804

Best: k=40, coherence=0.5804


In [5]:
# Topic vector per review
def doc_topic_vec(bow, n_topics):
    vec = np.zeros(n_topics)
    for tid, prob in best_lda.get_document_topics(bow, minimum_probability=0):
        vec[tid] = prob
    return vec

n = best_n
df["topic_vec"] = [doc_topic_vec(bow, n) for bow in corpus]

# Aggregate lên book level (mean)
book_topics = (
    df.groupby("book_id")["topic_vec"]
    .apply(lambda vecs: np.mean(np.stack(vecs), axis=0))
    .reset_index()
)
book_topics.columns = ["book_id", "topic_distribution"]

# Expand thành columns để dễ dùng
topic_cols = pd.DataFrame(
    book_topics["topic_distribution"].tolist(),
    columns=[f"t{i}" for i in range(n)]
)
book_topic_df = pd.concat([book_topics["book_id"], topic_cols], axis=1)
print(book_topic_df.shape)
print(book_topic_df.head(2))

(447813, 41)
  book_id        t0        t1        t2        t3        t4        t5  \
0       1  0.003103  0.004492  0.031583  0.047417  0.003627  0.008193   
1    1000  0.000435  0.041520  0.000435  0.000435  0.000435  0.000435   

         t6        t7        t8  ...       t30       t31       t32       t33  \
0  0.001475  0.001634  0.001605  ...  0.001656  0.007217  0.035575  0.016331   
1  0.000435  0.000435  0.000435  ...  0.000435  0.000435  0.000435  0.000435   

        t34       t35       t36       t37       t38       t39  
0  0.001470  0.002645  0.022559  0.010063  0.002958  0.022703  
1  0.000435  0.000435  0.000435  0.141442  0.000435  0.000435  

[2 rows x 41 columns]


In [6]:
out = "/kaggle/working/lda/book_topic_vectors.parquet"
book_topic_df.to_parquet(out, index=False)

api.upload_file(
    path_or_fileobj=out,
    path_in_repo="lda/book_topic_vectors.parquet",
    repo_id=HF_REPO,
    repo_type="dataset",
    token=HF_TOKEN
)

# Upload top terms per topic (interpretability)
top_terms = []
for tid in range(n):
    terms = [w for w, _ in best_lda.show_topic(tid, topn=10)]
    top_terms.append({"topic_id": tid, "top_terms": terms})

pd.DataFrame(top_terms).to_parquet(
    "/kaggle/working/lda/topic_terms.parquet", index=False
)
api.upload_file(
    path_or_fileobj="/kaggle/working/lda/topic_terms.parquet",
    path_in_repo="lda/topic_terms.parquet",
    repo_id=HF_REPO,
    repo_type="dataset",
    token=HF_TOKEN
)
print("Done!")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Done!
